In [1]:
import os

from dotenv import load_dotenv

import daft
from daft import col, lit
from daft.functions import download, format
from daft.io import IOConfig, S3Config
from daft.unity_catalog import UnityCatalog

load_dotenv()

True

In [2]:
daft.set_planning_config(
    default_io_config=IOConfig(
        s3=S3Config(
            region_name="us-west-2",
            key_id=os.getenv("AWS_ACCESS_KEY_ID"),
            access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
            # session_token=os.getenv("AWS_SESSION_TOKEN")
        )
    )
)

DaftContext(_ctx=<builtins.PyDaftContext object at 0x10fbb2010>)

In [3]:
SOURCE_URI = "s3://daft-public-datasets/reddit-irl/source"
COMMENTS_URI = "s3://daft-public-datasets/reddit-irl/comments.parquet"
qe_uri = "s3://daft-public-datasets/reddit-irl/embeddings/Qwen--Qwen3-Embedding-0.6B.parquet/*.parquet"
reviews_uri = "s3://srinu-dump/amazon-reviews/"
DEST_URI = "s3://daft-public-datasets/reddit-irl/all_images"
INDEX_URI = f"{DEST_URI}/_reddit_irl_images_index.parquet"

In [5]:
unity = UnityCatalog(
    endpoint=os.getenv("DATABRICKS_ENDPOINT"),
    token=os.getenv("DATABRICKS_TOKEN"),
)

catalog = daft.Catalog.from_unity(unity)

table = catalog.get_table("jaytest-unity.demo.reddit_irl_images_index")
print(table)

Table('reddit_irl_images_index')


In [7]:
IMAGES_URI = "s3://srinu-dump/conceptual-captions-12m-webdataset-images-v3/"
IMAGES_INDEX_URI = f"{IMAGES_URI}/_all_images_index.parquet"
files_df = daft.from_glob_path(f"{IMAGES_URI}/*.png").write_parquet(f"{IMAGES_URI}/_all_images_index.parquet")

Error when running pipeline node GlobScanSource
RuntimeStatsManager finished with active nodes {0}


FileNotFoundError: File: s3://srinu-dump/conceptual-captions-12m-webdataset-images-v3//*.png not found
No files found

In [4]:
df = daft.read_parquet(COMMENTS_URI)

df.show()

idString,id_xxhashUInt64,created_utcTimestamp[ms; UTC],subreddit_idString,subreddit_nameString,subreddit_nsfwBool,permalinkString,bodyString,sentimentFloat32,scoreInt32
d6ogcjv,4592170193240941746,2016-08-19 19:41:49 +00:00,2vegg,me_irl,false,https://old.reddit.com/r/me_irl/comments/4yitgg/meirl/d6ogcjv/,im 2 scared,None,2
d6ogcc8,11061850444114282782,2016-08-19 19:41:42 +00:00,2vegg,me_irl,false,https://old.reddit.com/r/me_irl/comments/4yicas/me_irl/d6ogcc8/,"That sounds delightful, tbh.",0.5859,3
d6ogbyg,10756139398361333582,2016-08-19 19:41:29 +00:00,2vegg,me_irl,false,https://old.reddit.com/r/me_irl/comments/4yicas/me_irl/d6ogbyg/,[removed],None,1
d6ogajy,17849396790374664025,2016-08-19 19:40:43 +00:00,2vegg,me_irl,false,https://old.reddit.com/r/me_irl/comments/4yicas/me_irl/d6ogajy/,Don't look at me or my bucket again,0,62
d6oga5j,13604117654023775118,2016-08-19 19:40:29 +00:00,2vegg,me_irl,false,https://old.reddit.com/r/me_irl/comments/4yicas/me_irl/d6oga5j/,"My God, I didn't even notice that tidbit.",0.2732,31
d6og9cr,4370743280362014639,2016-08-19 19:40:02 +00:00,2vegg,me_irl,false,https://old.reddit.com/r/me_irl/comments/4yicas/me_irl/d6og9cr/,Dun dun,None,18
d6og90i,13181282969615525994,2016-08-19 19:39:51 +00:00,2vegg,me_irl,false,https://old.reddit.com/r/me_irl/comments/4yk6ko/me_irl/d6og90i/,Guess we'll never find out*zing*,0,2
d6og8xh,5987077697040584183,2016-08-19 19:39:48 +00:00,2vegg,me_irl,false,https://old.reddit.com/r/me_irl/comments/4yicas/me_irl/d6og8xh/,[removed],None,1


🗡️ 🐟 Parquet Scan: 00:00 

In [ ]:
df = daft.read_parquet("s3://srinu-dump/amazon-reviews-embeddings/_stats2.parquet").sort("rows_per_second", desc=True)

df.show()

timestampFloat64,model_nameString,embedding_dimensionsInt64,rows_processedInt64,processing_timeFloat64,batch_sizeInt64,rows_per_secondFloat64
1765916593.584907,Qwen/Qwen3-Embedding-0.6B,1024,10000,23.23515748977661,64,430.3822775636432
1765916682.4885297,Qwen/Qwen3-Embedding-0.6B,1024,10000,23.897740602493286,32,418.44960016666533
1765916732.1795166,Qwen/Qwen3-Embedding-0.6B,1024,10000,26.316681385040283,16,379.9871212365135
1765915746.988116,Qwen/Qwen3-Embedding-0.6B,1024,10000,28.208296298980713,256,354.50563529288166
1765916330.4654408,Qwen/Qwen3-Embedding-0.6B,1024,10000,47.6172616481781,128,210.00787642694303
1765916080.6906126,Qwen/Qwen3-Embedding-0.6B,1024,40000,196.05966877937317,512,204.01952247003018


In [17]:
files_df = daft.from_glob_path(f"{DEST_URI}/*.png")

# Extract metadata from path and include file info from glob
index_df = (
    files_df.with_column("id", col("path").regexp_extract(r"_id([a-zA-Z0-9]+)\.png$", 1)).with_column(
        "image_xxhash",
        col("path").regexp_extract(r"_xxhash([0-9]+)_id", 1).cast(daft.DataType.uint64()),
    )
).show()

pathString,sizeInt64,num_rowsInt64,idString,image_xxhashUInt64
s3://daft-public-datasets/reddit-irl/all_images/reddit-irl_xxhash10000036941492261487_id42m146.png,30054,None,42m146,10000036941492261487
s3://daft-public-datasets/reddit-irl/all_images/reddit-irl_xxhash10000043170510662876_idsxcgpy.png,135550,None,sxcgpy,10000043170510662876
s3://daft-public-datasets/reddit-irl/all_images/reddit-irl_xxhash10000068230400297118_id4isk1y.png,63412,None,4isk1y,10000068230400297118
s3://daft-public-datasets/reddit-irl/all_images/reddit-irl_xxhash10000094888701977341_id7dyzs7.png,15826,None,7dyzs7,10000094888701977341
s3://daft-public-datasets/reddit-irl/all_images/reddit-irl_xxhash1000012259463533613_id45skuy.png,7517,None,45skuy,1000012259463533613
s3://daft-public-datasets/reddit-irl/all_images/reddit-irl_xxhash10000124789420064480_idpljtgr.png,14626,None,pljtgr,10000124789420064480
s3://daft-public-datasets/reddit-irl/all_images/reddit-irl_xxhash1000016592192531746_idfx1ldc.png,26191,None,fx1ldc,1000016592192531746
s3://daft-public-datasets/reddit-irl/all_images/reddit-irl_xxhash10000187067773025777_idb0feie.png,74173,None,b0feie,10000187067773025777


In [ ]:
SOURCE_URI = "s3://daft-public-datasets/reddit-irl"


df = daft.read_parquet(f"{SOURCE_URI}/*.parquet")

df = (
    df.with_column("created_utc", col("created_utc").cast(daft.DataType.timestamp("ms", "UTC")))
    .where(col("url").length() > 0)
    .with_column("bytes", download(daft.col("url"), on_error="null"))
    .where(col("bytes").not_null())
    .with_column("image_path", format("{}/{}.png", lit(SOURCE_URI), lit("image")))
    .with_column("image_written", image_writer.write_images(col("bytes"), col("image_path")))
)
df.limit(10).show()

typeString,idString,subreddit.idString,subreddit.nameString,subreddit.nsfwBool,created_utcTimestamp[ms; UTC],permalinkString,domainString,urlString,selftextString,titleString,scoreInt32,bytesBinary,image_pathString,image_writtenString
post,ttd04u,2s5ti,meirl,false,2022-03-31 23:44:02 +00:00,https://old.reddit.com/r/meirl/comments/ttd04u/meirl/,i.redd.it,https://i.redd.it/p86ehgxw2tq81.jpg,,Meirl,605,"b""\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01""...",s3://daft-public-datasets/reddit-irl/image.png,s3://daft-public-datasets/reddit-irl/image.png
post,ttc58g,2s5ti,meirl,false,2022-03-31 23:00:00 +00:00,https://old.reddit.com/r/meirl/comments/ttc58g/me_irl/,i.redd.it,https://i.redd.it/to2ot762vsq81.jpg,,me irl,32,"b""\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01""...",s3://daft-public-datasets/reddit-irl/image.png,s3://daft-public-datasets/reddit-irl/image.png
post,ttc1qh,2s5ti,meirl,false,2022-03-31 22:54:59 +00:00,https://old.reddit.com/r/meirl/comments/ttc1qh/meirl/,i.redd.it,https://i.redd.it/rxur9tl5usq81.jpg,,meirl,34,"b""\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01""...",s3://daft-public-datasets/reddit-irl/image.png,s3://daft-public-datasets/reddit-irl/image.png
post,ttc7og,2s5ti,meirl,false,2022-03-31 23:02:56 +00:00,https://old.reddit.com/r/meirl/comments/ttc7og/meirl/,imgur.com,https://imgur.com/KIU84VZ.jpeg,,meirl,67,"b""\xff\xd8\xff\xdb\x00\x84\x00\x05\x05""...",s3://daft-public-datasets/reddit-irl/image.png,s3://daft-public-datasets/reddit-irl/image.png
post,ttdab1,2s5ti,meirl,false,2022-03-31 23:59:09 +00:00,https://old.reddit.com/r/meirl/comments/ttdab1/meirl/,i.imgur.com,https://i.imgur.com/ucZiw34.jpg,,meirl,93,"b""\xff\xd8\xff\xdb\x00C\x00\x01\x01\x01""...",s3://daft-public-datasets/reddit-irl/image.png,s3://daft-public-datasets/reddit-irl/image.png
post,ttb8c3,2s5ti,meirl,false,2022-03-31 22:14:15 +00:00,https://old.reddit.com/r/meirl/comments/ttb8c3/me_irl/,i.redd.it,https://i.redd.it/990cmy4wmsq81.jpg,,Me irl,37,"b""\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01""...",s3://daft-public-datasets/reddit-irl/image.png,s3://daft-public-datasets/reddit-irl/image.png
post,ttd0r4,2s5ti,meirl,false,2022-03-31 23:44:56 +00:00,https://old.reddit.com/r/meirl/comments/ttd0r4/me_irl/,i.redd.it,https://i.redd.it/x8sehlq23tq81.jpg,,Me_irl,1,"b""\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01""...",s3://daft-public-datasets/reddit-irl/image.png,s3://daft-public-datasets/reddit-irl/image.png
post,ttbv93,2s5ti,meirl,false,2022-03-31 22:45:39 +00:00,https://old.reddit.com/r/meirl/comments/ttbv93/meirl/,i.redd.it,https://i.redd.it/bxfvfw1issq81.jpg,,Meirl,144,"b""\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01""...",s3://daft-public-datasets/reddit-irl/image.png,s3://daft-public-datasets/reddit-irl/image.png


In [ ]:
columns = [
    "type",
    "id",
    "subreddit.id",
    "subreddit.name",
    "subreddit.nsfw",
    "title",
    "created_utc",
    "permalink",
    "domain",
    "score",
    col("url").alias("image_url"),
    col("image_written").alias("image_s3_uri"),
]
df.select(columns).write_parquet(f"{DEST_URI}/_reddit_irl_index.parquet")

In [27]:
df = daft.read_parquet(COMMENTS_URI)
df.show()

DaftCoreException: DaftError::External Unhandled Error for path: s3://daft-public-datasets/reddit-irl/comments.parquet
Details:
unhandled error: Error { s3_extended_request_id: "WSE2Rc3ifr2EQBOzPF1mRhlhALux5GCCSxviSmayK6jK3zCAdAG5zwSVQAX6IlNZbAHuB26L3Fw=", aws_request_id: "ER9SW7Z963HEF86D" } (Unhandled(Unhandled { source: ErrorMetadata { code: None, message: None, extras: Some({"s3_extended_request_id": "WSE2Rc3ifr2EQBOzPF1mRhlhALux5GCCSxviSmayK6jK3zCAdAG5zwSVQAX6IlNZbAHuB26L3Fw=", "aws_request_id": "ER9SW7Z963HEF86D"}) }, meta: ErrorMetadata { code: None, message: None, extras: None } }))